# Setup: prepare the Lakehouse demo data
Run this notebook before notebooks 1-4. Use Pure Python and attach a **schema-enabled** Lakehouse named `lh_pond`.
<br>The standard demo uses 2025 flight Parquet files and airport reference CSVs. Later setup cells create a Delta table and an optional Excel workbook.

## 1. Choose the data scope
**Use `DATA_SCOPE = "one_year"` for this session.** It selects January-December 2025 and prints the workspace, Lakehouse and output paths.
<br>`full` prepares 2022-2025 for separate validation; the attendee demos still read `one_year`.

In [ ]:
from datetime import datetime
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import urlretrieve
import csv

import duckdb
import notebookutils

DATA_SCOPE = "one_year"  # Standard session data; choose "full" for separate validation.

DATA_RANGES = {
    "one_year": ("2025-01", "2025-12"),
    "full": ("2022-01", "2025-12"),
}

if DATA_SCOPE not in DATA_RANGES:
    valid_scopes = ", ".join(sorted(DATA_RANGES))
    raise ValueError(f"DATA_SCOPE must be one of: {valid_scopes}")

START_MONTH, END_MONTH = DATA_RANGES[DATA_SCOPE]

LAKEHOUSE = "/lakehouse/default"
OUTPUT_DIR = Path(f"{LAKEHOUSE}/Files/flights_eu/{DATA_SCOPE}")
REFERENCE_DIR = Path(f"{LAKEHOUSE}/Files/flights_eu/reference")
SCHEMA_DRIFT_DIR = OUTPUT_DIR / "schema_drift"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
SCHEMA_DRIFT_DIR.mkdir(parents=True, exist_ok=True)

OPDI_URL_TEMPLATE = (
    "https://www.eurocontrol.int/performance/data/download/"
    "OPDI/v002/flight_list/flight_list_{yyyymm}.parquet"
)
AIRPORTS_URL = "https://davidmegginson.github.io/ourairports-data/airports.csv"
AIRLINES_URL = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airlines.dat"
AIRLINES_COLUMNS = [
    "airline_id",
    "name",
    "alias",
    "iata_code",
    "icao_code",
    "callsign",
    "country",
    "active",
]

print("Data scope:", DATA_SCOPE)
print("Date range:", START_MONTH, "to", END_MONTH)
print("Workspace:", notebookutils.runtime.context["currentWorkspaceName"])
print("Lakehouse:", notebookutils.runtime.context["defaultLakehouseName"])
print("Output:", OUTPUT_DIR)
print("Reference data:", REFERENCE_DIR)
print("Schema-drift demo data:", SCHEMA_DRIFT_DIR)

## 2. Define the download helpers
**Define the functions; downloads start in the next cell.** Existing non-empty files are skipped, and failed downloads are removed.
<br>The airline helper converts the downloaded reference file into a CSV with column names.

In [ ]:
def month_range(start: str, end: str):
    start_dt = datetime.strptime(start, "%Y-%m")
    end_dt = datetime.strptime(end, "%Y-%m")
    if end_dt < start_dt:
        raise ValueError(f"END_MONTH ({end}) is before START_MONTH ({start})")

    year, month = start_dt.year, start_dt.month
    while (year, month) <= (end_dt.year, end_dt.month):
        yield f"{year}{month:02d}"
        month += 1
        if month > 12:
            month = 1
            year += 1


def download(url: str, dest: Path) -> bool:
    if dest.exists() and dest.stat().st_size > 0:
        print(f"skip: {dest.name}")
        return True

    try:
        print(f"download: {dest.name} ...", end=" ", flush=True)
        urlretrieve(url, dest)
        size_mb = dest.stat().st_size / (1024 * 1024)
        print(f"done ({size_mb:.1f} MB)")
        return True
    except HTTPError as error:
        print(f"failed HTTP {error.code}: {url}")
    except URLError as error:
        print(f"failed {error.reason}: {url}")

    if dest.exists():
        dest.unlink()
    return False


def download_airlines(dest: Path) -> bool:
    if dest.exists() and dest.stat().st_size > 0:
        print(f"skip: {dest.name}")
        return True

    raw_dest = dest.with_suffix(".dat")
    if not download(AIRLINES_URL, raw_dest):
        return False

    with open(raw_dest, newline="", encoding="utf-8") as source_file:
        rows = list(csv.reader(source_file))

    with open(dest, "w", newline="", encoding="utf-8") as target_file:
        writer = csv.writer(target_file)
        writer.writerow(AIRLINES_COLUMNS)
        for row in rows:
            writer.writerow(["" if value == r"\N" else value for value in row])

    raw_dest.unlink(missing_ok=True)
    print(f"converted: {raw_dest.name} -> {dest.name}")
    return True

## 3. Download the flight and reference files
**Download the selected months and shared lookup CSVs into Lakehouse Files.** The cell stops with a list of missing files if a download fails.
<br>Flight files go into `Files/flights_eu/{DATA_SCOPE}`; reference files go into `Files/flights_eu/reference`.

In [ ]:
months = list(month_range(START_MONTH, END_MONTH))
print(f"Pulling {len(months)} month(s): {START_MONTH} -> {END_MONTH}")
print()

failures = []
for yyyymm in months:
    url = OPDI_URL_TEMPLATE.format(yyyymm=yyyymm)
    dest = OUTPUT_DIR / f"flight_list_{yyyymm}.parquet"
    if not download(url, dest):
        failures.append(yyyymm)

print()
print("Reference data:")
airports_ok = download(AIRPORTS_URL, REFERENCE_DIR / "airports.csv")
airlines_ok = download_airlines(REFERENCE_DIR / "airlines.csv")

if failures:
    raise RuntimeError(f"Missing months: {', '.join(failures)}")
if not airports_ok or not airlines_ok:
    missing_references = []
    if not airports_ok:
        missing_references.append("airports.csv")
    if not airlines_ok:
        missing_references.append("airlines.csv")
    raise RuntimeError(f"Missing reference data: {', '.join(missing_references)}")

print("Setup download complete.")

## 4. Verify the download and count flight rows
**Check that every expected monthly file exists, then count the rows with DuckDB.** Missing or unexpected month files stop the check.
<br>This verifies file names, non-empty reference files and queryability; it is not a full data-quality audit.

In [ ]:
con = duckdb.connect()
flight_files = f"{OUTPUT_DIR}/flight_list_*.parquet"
expected_filenames = {
    f"flight_list_{yyyymm}.parquet"
    for yyyymm in month_range(START_MONTH, END_MONTH)
}
actual_filenames = {path.name for path in OUTPUT_DIR.glob("flight_list_*.parquet")}
assert actual_filenames == expected_filenames, (
    "Flight files do not match the selected scope. "
    f"Missing: {sorted(expected_filenames - actual_filenames)}; "
    f"unexpected: {sorted(actual_filenames - expected_filenames)}"
)
assert (REFERENCE_DIR / "airports.csv").stat().st_size > 0, "airports.csv is missing or empty"
assert (REFERENCE_DIR / "airlines.csv").stat().st_size > 0, "airlines.csv is missing or empty"

flight_count = con.sql(f"""
    SELECT count(*) AS flights
    FROM read_parquet('{flight_files}')
""").df()

display(flight_count)

## 4a. Check the count for each monthly file
**List the row count in each downloaded Parquet file.** This makes missing or unexpectedly small months easier to spot.

In [ ]:
con.sql(f"""
    SELECT filename,
           count(*) AS rows_in_file
    FROM read_parquet('{flight_files}', filename = true)
    GROUP BY 1
    ORDER BY 1
""").df()

## 5. Create the January Delta table for notebook 4
**Write January's flight data to `dbo.flights_202501`.** Notebook 4 reads it with `delta_scan` and compares its Barcelona count with the original Parquet file.
<br>This overwrites that derived table through OneLake. The source files in Files are unchanged.

In [ ]:
from deltalake import DeltaTable, write_deltalake

ctx = notebookutils.runtime.context

workspace_id = ctx["defaultLakehouseWorkspaceId"]
lakehouse_id = ctx["defaultLakehouseId"]

assert workspace_id and lakehouse_id, (
    "Attach a default schema-enabled Lakehouse before running this notebook."
)

TABLE_SCHEMA = "dbo"
TABLE_NAME = "flights_202501"

table_path = (
    f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/"
    f"{lakehouse_id}/Tables/{TABLE_SCHEMA}/{TABLE_NAME}"
)

storage_options = {
    "bearer_token": notebookutils.credentials.getToken("storage"),
    "use_fabric_endpoint": "true",
}

january_file = OUTPUT_DIR / "flight_list_202501.parquet"
assert january_file.exists(), f"January source file is missing: {january_file}"

january_flights = con.sql(f"""
    SELECT adep,
           ades,
           typecode,
           cast(dof AS varchar) AS dof
    FROM read_parquet('{january_file}')
""").arrow()

write_deltalake(
    table_path,
    january_flights,
    mode="overwrite",
    storage_options=storage_options,
)

delta_table = DeltaTable(table_path, storage_options=storage_options)

print(
    f"Created Delta table: {TABLE_SCHEMA}.{TABLE_NAME} "
    f"(version {delta_table.version()})"
)

## 6. Optional exercise: create schema-drift examples
**Create two small Parquet exports; the second adds `source_batch`.** The displayed `UNION ALL BY NAME` aligns their columns and fills the missing field with NULL.
<br>This is an extra setup exercise, not a dependency of notebook 2's examples. It replaces only these derived fixture files.

In [ ]:
schema_drift_files = {
    "earlier": SCHEMA_DRIFT_DIR / "flights_earlier.parquet",
    "later": SCHEMA_DRIFT_DIR / "flights_later.parquet",
}

for path in schema_drift_files.values():
    path.unlink(missing_ok=True)

con.sql(f"""
    COPY (
        SELECT adep,
               ades,
               typecode,
               dof
         FROM read_parquet('{flight_files}')
        WHERE adep IS NOT NULL
          AND ades IS NOT NULL
        LIMIT 500
    ) TO '{schema_drift_files["earlier"]}' (FORMAT PARQUET)
""")

con.sql(f"""
    COPY (
        SELECT adep,
               ades,
               typecode,
               dof,
               '2025-refresh' AS source_batch
        FROM read_parquet('{flight_files}')
        WHERE adep IS NOT NULL
          AND ades IS NOT NULL
        LIMIT 500
    ) TO '{schema_drift_files["later"]}' (FORMAT PARQUET)
""")

assert all(path.exists() for path in schema_drift_files.values()), (
    "Schema-drift fixture files were not created."
)

con.sql(f"""
    SELECT *
    FROM (
        SELECT *
        FROM read_parquet('{schema_drift_files["earlier"]}')
        LIMIT 3
    )

    UNION ALL BY NAME

    SELECT *
    FROM (
        SELECT *
        FROM read_parquet('{schema_drift_files["later"]}')
        LIMIT 3
    )
""").df()

## 7. Create the workbook for notebook 4's optional Excel example
**Write January's Barcelona departures by destination to an `.xlsx` file, then read it back.** Its `flights` total should match the other January sources.
<br>Missing destinations are labelled `UNKNOWN` to keep the first text cell non-empty for type inference. This replaces the derived workbook.

In [ ]:
con.execute("INSTALL excel")
con.execute("LOAD excel")

EXCEL_DIR = Path(f"{LAKEHOUSE}/Files/flights_eu/excel")
EXCEL_DIR.mkdir(parents=True, exist_ok=True)
excel_file = EXCEL_DIR / "barcelona_departures_202501.xlsx"
excel_file.unlink(missing_ok=True)

con.sql(f"""
    COPY (
        SELECT coalesce(ades, 'UNKNOWN') AS destination,
               count(*) AS flights
        FROM read_parquet('{january_file}')
        WHERE adep = 'LEBL'
        GROUP BY ALL
        ORDER BY flights DESC
    ) TO '{excel_file}' WITH (FORMAT xlsx, HEADER true)
""")

assert excel_file.exists() and excel_file.stat().st_size > 0, "Excel workbook was not created."

con.sql(f"""
    SELECT count(*) AS destinations,
           CAST(sum(flights) AS BIGINT) AS flights_from_barcelona
    FROM read_xlsx('{excel_file}')
""").df()

## Next
Open `1. Query a Lakehouse File`. The demos read the `one_year` folder; prepare that scope even if you also run separate full-data validation.